# Phase 14 — Streamlit UI (headless smoke test)

> Run AFTER Phase 14 (restart kernel first). Drives the EXACT same graph the UI calls (`GraphBuilder().with_all().build()` + `astream_events`), so you can verify the end-to-end path without launching a browser.

**Acceptance:** pick CLT-005, ask a technical-analysis question (CLT-005 holds NVDA), confirm agent events stream and citations/portfolio data are extractable exactly as the UI components render them.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. Build the graph exactly as the UI does (Facade)

In [2]:
import aiosqlite
from pathlib import Path
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from app.graph.builder import GraphBuilder
from app.config import settings

# astream_events() needs an async-capable checkpointer — the default sync
# SqliteSaver raises NotImplementedError under it (found + fixed the same
# issue in the Streamlit UI). Anchor the path to the project root — the
# notebook's cwd is notebooks/, and the relative default would otherwise
# resolve to notebooks/.checkpoints/... (the same class of bug fixed for
# the Excel repository back in Phase 1).
_ckpt_path = Path("..") / settings.SQLITE_CHECKPOINT_PATH
_ckpt_path.parent.mkdir(parents=True, exist_ok=True)
_conn = await aiosqlite.connect(str(_ckpt_path))
_checkpointer = AsyncSqliteSaver(_conn)
await _checkpointer.setup()

graph = GraphBuilder().with_all(checkpointer=_checkpointer).build()
print("graph built —", len(list(graph.get_graph().nodes)), "nodes")


graph built — 15 nodes


## 2. Drive it with astream_events (same call the UI's stream_turn() makes)

In [3]:
import asyncio
from langchain_core.messages import HumanMessage, AIMessage
from app.agents.base import _text

cfg = {'configurable': {'thread_id': 'CLT-005-nb14'}}
run_input = {'messages': [HumanMessage(content='technical analysis on my NVDA position')],
             'client_id': 'CLT-005', 'session_id': 'nb14'}

async def drive():
    seen_nodes = []
    async for ev in graph.astream_events(run_input, cfg, version='v2'):
        if ev['event'] == 'on_chain_start' and ev.get('name'):
            seen_nodes.append(ev['name'])
    return seen_nodes

seen = await drive()
print('node path:', ' -> '.join(seen))

{"client_id": "CLT-005", "prior_decisions": 4, "event": "memory_read", "level": "info", "timestamp": "2026-07-13T13:38:07.633045Z"}


{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:38:07.634741Z"}


{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:38:07.635205Z"}


{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:38:07.635517Z"}


AFC is enabled with max remote calls: 10.


...

{"guard": "citation_coverage", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "reflector", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:40:10.977414Z"}


{"guard": "groundedness", "action": "pass", "passed": true, "reason": "no retrieved context", "event": "guardrail_check", "agent": "reflector", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:40:10.978367Z"}


{"revisions": 0, "event": "reflection_pass", "agent": "reflector", "client_id": "CLT-005", "session_id": "nb14", "level": "info", "timestamp": "2026-07-13T13:40:10.979325Z"}


{"client_id": "CLT-005", "session_id": "nb14", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T13:40:10.996193Z"}


node path: LangGraph -> memory_read -> input_guard -> planner -> clarifier -> supervisor -> portfolio -> LangGraph -> model -> tools -> model -> tools -> model -> supervisor -> securities_analysis -> LangGraph -> model -> tools -> model -> tools -> model -> supervisor -> synthesizer -> reflector -> memory_write


## 3. Pull the final answer + tool_results exactly as the UI's citations panel does

In [4]:
# aget_state (not get_state): the notebook kernel's single event loop IS the
# checkpointer's own loop, and AsyncSqliteSaver forbids sync calls from that
# same loop/thread -- must await the async equivalent here.
state = await graph.aget_state(cfg)
final = None
for m in reversed(state.values.get("messages", [])):
    if isinstance(m, AIMessage) and m.content:
        final = _text(m.content); break
print(final[:400])

from ui.components.citations_panel import extract_citations
citations = extract_citations(state.values.get("tool_results", {}))
print(f"{len(citations)} citation(s) extractable for the UI panel")
for c in citations[:3]:
    print(" -", c["label"])


### NVDA Position Analysis

**1. Client Request**
You requested a technical analysis of your current position in NVIDIA Corporation (NVDA).

**2. Portfolio Performance Summary**
As of July 13, 2026, your position in NVDA is as follows:
*   **Quantity:** 150 shares
*   **Current Price:** $208.92
*   **Purchase Price:** $21.578 (acquired February 14, 2023)
*   **Market Value:** $31,338.00
*   **Abso


0 citation(s) extractable for the UI panel


## 4. Portfolio panel data (same call ui/components/portfolio_panel.py makes)

In [5]:
from app.tools.portfolio_tools import get_allocation_by_asset_class, get_portfolio_value
alloc = get_allocation_by_asset_class('CLT-005')
value = get_portfolio_value('CLT-005')
print('total value:', value['total_value'])
print('allocation :', alloc['allocation_pct'])

{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T13:40:12.238944Z"}


{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T13:40:12.755229Z"}


{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T13:40:13.833537Z"}


{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T13:40:14.347234Z"}


{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-13T13:40:15.370456Z"}


...

{"fn": "_quote", "age_s": 4.3, "event": "cache_hit", "level": "info", "timestamp": "2026-07-13T13:40:25.822319Z"}


{"fn": "_quote", "age_s": 2.2, "event": "cache_hit", "level": "info", "timestamp": "2026-07-13T13:40:25.823329Z"}


{"fn": "_quote", "age_s": 1.1, "event": "cache_hit", "level": "info", "timestamp": "2026-07-13T13:40:25.824159Z"}


{"fn": "_quote", "age_s": 0.0, "event": "cache_hit", "level": "info", "timestamp": "2026-07-13T13:40:25.825016Z"}


total value: 1184341.7
allocation : {'Individual Stock': 95.51, 'Cash Equivalent': 4.49}


## 5. Confirm a portfolio-level question distinguishes stocks / ETFs / cash

In [6]:
cfg2 = {"configurable": {"thread_id": "CLT-005-nb14b"}}
run_input2 = {"messages": [HumanMessage(content="What do I own?")],
              "client_id": "CLT-005", "session_id": "nb14b"}
async def drive2():
    async for ev in graph.astream_events(run_input2, cfg2, version="v2"):
        pass
await drive2()
state2 = await graph.aget_state(cfg2)
answer2 = next(_text(m.content) for m in reversed(state2.values["messages"])
               if isinstance(m, AIMessage) and m.content)
print(answer2[:500])


{"client_id": "CLT-005", "prior_decisions": 5, "event": "memory_read", "level": "info", "timestamp": "2026-07-13T13:40:25.836594Z"}


{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14b", "level": "info", "timestamp": "2026-07-13T13:40:25.838842Z"}


{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14b", "level": "info", "timestamp": "2026-07-13T13:40:25.839227Z"}


{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "agent": "input_guard", "client_id": "CLT-005", "session_id": "nb14b", "level": "info", "timestamp": "2026-07-13T13:40:25.839564Z"}


{"query": "What do I own?", "event": "planner_simple", "agent": "planner", "client_id": "CLT-005", "session_id": "nb14b", "level": "info", "timestamp": "2026-07-13T13:40:25.840937Z"}


...

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


{"guard": "citation_coverage", "action": "revise", "passed": false, "reason": "unsupported claims: ['The final summary table is incomplete and cuts off after listing Snowflake Inc (SNOW), failing to ", "event": "guardrail_check", "agent": "reflector", "client_id": "CLT-005", "session_id": "nb14b", "level": "info", "timestamp": "2026-07-13T13:41:35.391624Z"}


{"reason": "unsupported claims: ['The final summary table is incomplete and cuts off after listing Snowflake Inc (SNOW), failing to ", "event": "reflection_max_revisions", "agent": "reflector", "client_id": "CLT-005", "session_id": "nb14b", "level": "warning", "timestamp": "2026-07-13T13:41:35.392439Z"}


{"client_id": "CLT-005", "session_id": "nb14b", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T13:41:35.409060Z"}


### Analysis of Your Portfolio

**1. Client Request**
You requested a summary of the assets currently held in your portfolio.

**2. Evidence Review**
Based on the provided portfolio data, your holdings consist of 12 distinct assets: 11 individual stocks and one cash equivalent.

*   **Individual Stocks:**
    *   **Apple Inc (AAPL):** 850 shares (Purchased at $121.41; Sector: Technology)
    *   **Microsoft Corporation (MSFT):** 400 shares (Purchased at $209.54; Sector: Technology)
    *   **Alp


## ✅ Acceptance check

In [7]:
assert 'securities_analysis' in seen or 'portfolio' in seen
assert final
low = answer2.lower()
assert any(k in low for k in ['stock', 'individual']) and 'cash' in low
print('Phase 14 acceptance: PASS — headless graph path matches what the UI drives')

Phase 14 acceptance: PASS — headless graph path matches what the UI drives
